# 02 — Preprocesado clásico (CLAHE)

Probamos el módulo `src/preprocessing` sobre imágenes reales del dataset Freiburg Groceries.

**Objetivo**: ver visualmente que CLAHE iguala el contraste local sin alterar los colores, y comparar el efecto con distintos `clip_limit`.

**Pre-requisitos**: haber ejecutado `python scripts/download_freiburg.py`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import apply_clahe, preprocess
from src.utils.io_utils import load_image, list_images

DATA_ROOT = Path('../data/external/klasson_flat')
plt.rcParams['figure.dpi'] = 90

## 1. Comprobar que el dataset está descargado

In [ ]:
if not DATA_ROOT.is_dir():
    print(f'❌  No se encuentra {DATA_ROOT}')
    print('Ejecuta primero: python scripts/download_freiburg.py')
else:
    categories = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
    print(f'✓  Dataset listo. Categorías encontradas ({len(categories)}):')
    for c in categories:
        n = len(list_images(DATA_ROOT / c))
        print(f'    {c:25s} {n:4d} imágenes')

## 2. Antes / después en una sola imagen

Cogemos una imagen al azar y aplicamos CLAHE con los parámetros por defecto.

In [ ]:
# Cambia el nombre de la categoría si quieres probar con otra
category = 'BEANS'
images = list_images(DATA_ROOT / category)

if not images:
    print(f'No hay imágenes en {category}, prueba con otra categoría.')
else:
    img = load_image(images[0])
    img_clahe = preprocess(img)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img)
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(img_clahe)
    axes[1].set_title('CLAHE (clip=2.0, grid=8x8)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

## 3. Comparativa de `clip_limit`

Probamos cómo cambia el resultado al variar el parámetro de contraste. Más alto → más contraste, pero también más ruido amplificado.

In [ ]:
clip_limits = [1.0, 2.0, 4.0, 8.0]

fig, axes = plt.subplots(1, len(clip_limits) + 1, figsize=(4 * (len(clip_limits) + 1), 4))
axes[0].imshow(img)
axes[0].set_title('Original')
axes[0].axis('off')

for ax, cl in zip(axes[1:], clip_limits):
    out = apply_clahe(img, clip_limit=cl)
    ax.imshow(out)
    ax.set_title(f'clip_limit = {cl}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Histogramas: ver el efecto cuantitativamente

Sobre el canal de luminosidad (L). CLAHE redistribuye los valores hacia un rango más amplio.

In [ ]:
def get_l_channel(rgb_img):
    bgr = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    return lab[:, :, 0]

l_orig = get_l_channel(img)
l_clahe = get_l_channel(img_clahe)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(l_orig.ravel(), bins=50, alpha=0.5, label='Original', color='steelblue')
ax.hist(l_clahe.ravel(), bins=50, alpha=0.5, label='CLAHE', color='coral')
ax.set_xlabel('Valor de L (luminosidad)')
ax.set_ylabel('Frecuencia')
ax.set_title('Histograma del canal L antes / después')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Una galería rápida

Para que veas el efecto en varias imágenes a la vez.

In [ ]:
# Coger una imagen de cada una de las primeras 6 categorías
samples = []
for c in categories[:6]:
    imgs = list_images(DATA_ROOT / c)
    if imgs:
        samples.append((c, imgs[0]))

fig, axes = plt.subplots(len(samples), 2, figsize=(8, 3 * len(samples)))
for i, (cat, path) in enumerate(samples):
    img = load_image(path)
    img_pre = preprocess(img)
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'{cat} — original', fontsize=10)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(img_pre)
    axes[i, 1].set_title(f'{cat} — CLAHE', fontsize=10)
    axes[i, 1].axis('off')
plt.tight_layout()
plt.show()

## 6. Conclusiones

Cosas a observar al ejecutar este notebook:

- **Colores conservados**: la imagen procesada mantiene los tonos originales. Es lo que queríamos: solo igualar el contraste, no cambiar el color.
- **Más detalle local**: zonas oscuras se ven mejor sin quemar las claras.
- **Cuidado con `clip_limit` alto**: a partir de 4-8 empieza a amplificar ruido y los colores pueden quedar artificiales.

**Próximo paso**: `03_region_proposal.ipynb` para ver cómo segmentar regiones por color.